In [1]:
import pandas as pd 
import numpy as np 
import os 
import torch
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import torch.nn.functional as F
import clip
import glob

/l/vision/zekrom_ssd/fraramir/miniconda3/envs/world/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
compatibility_columns = ["subj", "img1", "img2", "category", "valid", "hist_corr"]
audit_required_columns = compatibility_columns + ["shuffle_img1", "shuffle_img2"]

def get_cosine_similarity(t1, t2):
    if t1.shape[0] != 1 or len(t1.shape) != 2:
        print(t1.shape)
    
    if t2.shape[0] != 1 or len(t2.shape) != 2:
        print(t2.shape)

    assert len(t1.shape) == 2
    assert len(t2.shape) == 2
    assert t1.shape[0] == 1
    assert t2.shape[0] == 1

    return F.cosine_similarity(t1, t2).item()

def return_image_set(df, img1_col="img1", img2_col="img2"):
    return set(df[img1_col]).union(set(df[img2_col]))

def require_columns(df, required_columns, name):
    missing = set(required_columns).difference(df.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

In [3]:
cats = ["bottle", "bowl", "chair", "cup", "door", "spoon", "table", "window"]
rgb_dir = "../../data/PanelA/RGB_hist"
clip_out_dir = "../../data/PanelA/deep_features/PE/PE-Spatial"
# audit_path = os.path.join(rgb_dir, "NULL_all2all_corr_shuffled_audit.csv")
audit_path = os.path.join(rgb_dir, "NULL_all2all_corr_shuffled_chunkedK1_audit.csv")

if not os.path.exists(audit_path):
    raise FileNotFoundError(f"Missing shuffled-null audit CSV: {audit_path}")

audit_df = pd.read_csv(audit_path)
require_columns(audit_df, audit_required_columns, "audit_df")

split_dfs = {}
for cat in cats:
    split_path = os.path.join(rgb_dir, f"NULL_forRplots_shuffled_chunkedK1_{cat}.csv")
    temp_df = pd.read_csv(split_path, header=None, names=compatibility_columns)
    split_dfs[cat] = temp_df

df = pd.concat(split_dfs.values(), ignore_index=True)
original_img_pool = return_image_set(df)
audit_original_img_pool = return_image_set(audit_df)
if original_img_pool != audit_original_img_pool:
    raise AssertionError(
        "Split null CSV graph-facing image pool does not match audit original image pool. "
        f"Only in split: {len(original_img_pool - audit_original_img_pool)}; "
        f"only in audit: {len(audit_original_img_pool - original_img_pool)}"
    )

all_shuffled_imgs = return_image_set(audit_df, "shuffle_img1", "shuffle_img2")
if len(all_shuffled_imgs) != len(original_img_pool):
    raise AssertionError(
        f"Shuffled image pool size ({len(all_shuffled_imgs)}) does not match "
        f"original/null image pool size ({len(original_img_pool)})."
    )

In [4]:
print(df.shape)
print()
print(df.head())

(1519564, 6)

    subj              img1              img2       category  valid  hist_corr
0  030FD  030FD_011375.jpg  030FD_012050.jpg  bottle_bottle   True   0.850039
1  030FD  030FD_011375.jpg  030FD_013975.jpg  bottle_bottle   True   0.691412
2  030FD  030FD_011375.jpg  030FD_014300.jpg  bottle_bottle   True   0.780458
3  030FD  030FD_011375.jpg  030FD_014400.jpg  bottle_bottle   True   0.853792
4  030FD  030FD_011375.jpg  030FD_015400.jpg  bottle_bottle   True   0.291637


In [5]:
print(f"Original/null image pool: {len(original_img_pool)}")
print(f"Shuffled feature image pool: {len(all_shuffled_imgs)}")

Original/null image pool: 6357
Shuffled feature image pool: 6357


In [ ]:
# NOTE
# Please follow the setup instructions in each linked repository

def extract_clip_features(target_imgs):
    """
    https://github.com/openai/CLIP
    """

    device = "cuda:7" if torch.cuda.is_available() else "cpu"
    print(device)
    model, preprocess = clip.load("ViT-B/32", device=device)

    folder_path = "/path/to/img" # NOTE images will be released on Databrary
    all_features = {}
    
    # for image_path in glob.glob(os.path.join(image_folder, "*.jpg")):
    for img_name in target_imgs:
        img_path = os.path.join(folder_path, img_name)
        image = preprocess(
                Image.open(img_path).convert("RGB")
            ).unsqueeze(0).to(device)

        with torch.no_grad():
            all_features[img_name] = model.encode_image(image).cpu()

    print(f"Extracted features for {len(all_features)} images.")

    return all_features

In [ ]:
def extract_pe_features(target_imgs):
    """
    https://github.com/facebookresearch/perception_models/tree/main
    """
    import core.vision_encoder.pe as pe
    import core.vision_encoder.transforms as transforms

    device = "cuda:7" if torch.cuda.is_available() else "cpu"
    
    folder_path = "/path/to/img" # NOTE images will be released on Databrary

    model = pe.VisionTransformer.from_config("PE-Spatial-L14-448", pretrained=True)
    model = model.to(device)
    preprocess = transforms.get_image_transform(model.image_size)

    all_features = {}

    for img_name in target_imgs:
        img_path = os.path.join(folder_path, img_name)
        image = Image.open(img_path).convert("RGB")
        
        with torch.no_grad():
            inputs = preprocess(image).unsqueeze(0).to(device)
            image_features, *_ = model(inputs)
        
            all_features[img_name] = image_features.cpu()

    print(f"Extracted features for {len(all_features)} images.")

    return all_features

In [8]:
# feats = extract_clip_features(all_shuffled_imgs)
feats = extract_pe_features(all_shuffled_imgs)

missing_features = all_shuffled_imgs.difference(feats)
if missing_features:
    raise AssertionError(
        f"Missing extracted features for {len(missing_features)} shuffled images. "
        f"First missing image: {sorted(missing_features)[0]}"
    )

Missing keys for loading vision encoder: []
Unexpected keys for loading vision encoder: []
Extracted features for 6357 images.


In [9]:
example_img = sorted(all_shuffled_imgs)[0]
print(example_img, feats[example_img].shape)
print(example_img, feats[example_img][0].unsqueeze(0).shape)

030FD_000025.jpg torch.Size([1025, 1024])
030FD_000025.jpg torch.Size([1, 1024])


In [10]:
os.makedirs(clip_out_dir, exist_ok=True)

for cat in cats:
    temp_df = split_dfs[cat].copy()
    original_columns = temp_df[compatibility_columns].copy()

    valid_mask = audit_df["valid"].astype(str).str.lower().eq("true")
    audit_cat = audit_df[(audit_df["category"] == f"{cat}_{cat}") & valid_mask].copy()

    if temp_df.shape[0] != audit_cat.shape[0]:
        raise AssertionError(
            f"Row count mismatch for {cat}: split has {temp_df.shape[0]}, "
            f"audit has {audit_cat.shape[0]}."
        )

    for col in compatibility_columns:
        split_col = temp_df[col].reset_index(drop=True)
        audit_col = audit_cat[col].reset_index(drop=True)
        # print(split_col.shape)
        # print(audit_col.shape)
        if col == "hist_corr":
            columns_match = np.allclose(split_col.astype(float), audit_col.astype(float))
        else:
            columns_match = split_col.equals(audit_col)
        if not columns_match:
            raise AssertionError(f"Split/audit row-order mismatch for {cat} column {col}.")

    temp_df["clip_cos_sim"] = [
        # get_cosine_similarity(feats[shuffle_img1], feats[shuffle_img2])
        get_cosine_similarity(feats[shuffle_img1][0].unsqueeze(0), feats[shuffle_img2][0].unsqueeze(0))
        for shuffle_img1, shuffle_img2 in zip(audit_cat["shuffle_img1"], audit_cat["shuffle_img2"])
    ]

    if temp_df.shape[0] != original_columns.shape[0]:
        raise AssertionError(f"Output row count changed for {cat}.")
    for col in compatibility_columns:
        if not temp_df[col].equals(original_columns[col]):
            raise AssertionError(f"Output compatibility column changed for {cat}: {col}.")

    out_path = os.path.join(clip_out_dir, f"NULL_forRplots_shuffled_chunkedK1_{cat}_withPE.csv")
    temp_df.to_csv(out_path, header=False, index=False)
    del temp_df
